# 05 · Gene Set Enrichment Analysis (GSEA)
**Input :** `data/processed/dea_results.csv`  
**Output:** `results/tables/gsea_go_bp.csv` · `gsea_go_mf.csv` · `gsea_go_cc.csv`
· `gsea_kegg.csv` · `results/figures/gsea_*.png`

Runs ranked GSEA via clusterProfiler (R) across GO-BP, GO-MF, GO-CC, and KEGG.
Highlights four HCC-relevant themes: lipid metabolism, glycolysis, PI3K-AKT/Wnt,
and immune regulation.

**Requires:** R packages `clusterProfiler`, `org.Hs.eg.db`, `enrichplot`.  
Install with: `Rscript env/r_packages.R`

**Run order:** 04 → **05** → P1


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

from paths import REPO_ROOT, RAW_DIR, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
print(f"Repo root : {REPO_ROOT}")
print(f"Raw data  : {RAW_DIR}")

In [ ]:
from gsea_functions import prepare_ranked_list, run_gsea_r, print_gsea_summary
print("Imports OK")

## 1 · R environment (run once per session)

In [ ]:
import os, sys

# ── R environment — update R_HOME if needed ───────────────────────────────────
# macOS  : /Library/Frameworks/R.framework/Resources
# Linux  : /usr/lib/R
# Windows: r"C:\Program Files\R\R-4.5.3"
R_HOME = os.environ.get("R_HOME", "")
if R_HOME:
    os.environ["R_HOME"] = R_HOME
    if sys.platform == "win32":
        os.environ["PATH"] = os.path.join(R_HOME,"bin","x64") + ";" + os.environ["PATH"]

%load_ext rpy2.ipython
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import default_converter
import anndata2ri
print("R environment ready")

## 2 · Install GSEA R packages (safe to re-run)

In [ ]:
%%R
pkgs <- c("clusterProfiler","org.Hs.eg.db","enrichplot","DOSE","ggplot2")
if (!requireNamespace("BiocManager", quietly=TRUE))
    install.packages("BiocManager", repos="https://cloud.r-project.org")
for (pkg in pkgs)
    if (!requireNamespace(pkg, quietly=TRUE))
        BiocManager::install(pkg, ask=FALSE, update=FALSE)
suppressPackageStartupMessages({
    library(clusterProfiler); library(org.Hs.eg.db)
    library(enrichplot);      library(ggplot2)
})
cat("GSEA packages ready\n")

## 3 · Prepare ranked gene list

In [ ]:
ranked = prepare_ranked_list(PROC_DIR / "dea_results.csv", PROC_DIR)
ranked.head(10)

## 4 · Run GSEA in R

In [ ]:
run_gsea_r(ro, PROC_DIR, FIGURES_DIR, TABLES_DIR)

## 5 · Results summary

In [ ]:
print_gsea_summary(TABLES_DIR)